# 🧹 Strict Data Cleaning + Advanced Feature Engineering

**Goal**: Remove sensor noise so XGBoost learns true physiological patterns  
**Method**: Polars Lazy API with streaming execution  
**Output**: Clean `train_ready_for_glucose_prediction.parquet`

In [ ]:
!pip install polars -U -q

import polars as pl
import numpy as np
import os

from google.colab import drive
drive.mount('/content/drive')

print(f"✅ Polars: {pl.__version__}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"

RAW_TRAIN = f"{BASE_PATH}/raw/metabonet_public_train.parquet"
OUTPUT_TRAIN = f"{BASE_PATH}/processed/train_ready_for_glucose_prediction.parquet"

os.makedirs(f"{BASE_PATH}/processed", exist_ok=True)

# Columns to load
KEEP_COLS = ['id', 'date', 'CGM', 'bolus', 'carbs', 'basal', 'age', 'weight', 'gender']

# Final output features
FINAL_FEATURES = [
    'CGM', 'bolus',
    'velocity', 'acceleration',
    'rolling_mean', 'rolling_std', 'active_insulin',
    'time_sin', 'time_cos',
    'age', 'weight', 'gender_encoded',
    'CGM_future'  # Target
]

# Cleaning thresholds
CGM_MIN = 39
CGM_MAX = 400
MAX_VELOCITY = 15  # Max allowed jump (mg/dL per 5 min)

print(f"📁 Input:  {RAW_TRAIN}")
print(f"📁 Output: {OUTPUT_TRAIN}")
print(f"\n🧹 Cleaning Rules:")
print(f"   CGM bounds: {CGM_MIN} < CGM < {CGM_MAX}")
print(f"   Max velocity: {MAX_VELOCITY} mg/dL per 5 min")

In [ ]:
print("="*70)
print("🔧 STRICT DATA CLEANING + ADVANCED FEATURE ENGINEERING")
print("="*70)

# ============================================================
# STEP 1: LOAD DATA (Lazy)
# ============================================================
print("\n📂 Step 1: Scanning parquet (lazy)...")
q = pl.scan_parquet(RAW_TRAIN)

# Get original count for stats
original_count = pl.scan_parquet(RAW_TRAIN).select(pl.len()).collect().item()
print(f"   Original rows: {original_count:,}")

# Select required columns
available = q.collect_schema().names()
keep = [c for c in KEEP_COLS if c in available]
print(f"   Columns: {keep}")
q = q.select(keep)

# ============================================================
# STEP 2: SORT BY PATIENT AND TIME
# ============================================================
print("\n📌 Step 2: Sorting by [id, date]...")
q = q.sort(['id', 'date'])

# ============================================================
# STEP A: BASIC FEATURES FOR CLEANING
# ============================================================
print("\n🔧 Step A: Creating features for cleaning...")

# Fill nulls first
q = q.with_columns([
    pl.col('CGM').fill_null(strategy='forward').over('id'),
    pl.col('bolus').fill_null(0.0),
    pl.col('carbs').fill_null(0.0),
    pl.col('basal').fill_null(0.0),
    pl.col('age').fill_null(-1.0),
    pl.col('weight').fill_null(-1.0),
])

# Create CGM_lag_1 and Velocity_Abs for cleaning
q = q.with_columns([
    pl.col('CGM').shift(1).over('id').alias('CGM_lag_1'),
])

q = q.with_columns([
    (pl.col('CGM') - pl.col('CGM_lag_1')).abs().alias('velocity_abs'),
])

# ============================================================
# STEP B: STRICT DATA CLEANING (THE FILTER)
# ============================================================
print("\n🧹 Step B: Applying STRICT cleaning filters...")

# Filter 1: Non-null CGM
print(f"   Filter 1: Remove null CGM")
q = q.filter(pl.col('CGM').is_not_null())

# Filter 2: Physiological bounds (39 < CGM < 400)
print(f"   Filter 2: CGM bounds ({CGM_MIN} < CGM < {CGM_MAX})")
q = q.filter(
    (pl.col('CGM') > CGM_MIN) & (pl.col('CGM') < CGM_MAX)
)

# Filter 3: Impossible jumps (velocity_abs < 15)
print(f"   Filter 3: Max velocity < {MAX_VELOCITY} mg/dL per 5min")
q = q.filter(
    (pl.col('velocity_abs') < MAX_VELOCITY) | pl.col('velocity_abs').is_null()
)

# ============================================================
# STEP C: ADVANCED FEATURE ENGINEERING (On Cleaned Data)
# ============================================================
print("\n🧬 Step C: Creating ADVANCED features...")

# Cast to Float32
q = q.with_columns([
    pl.col('CGM').cast(pl.Float32),
    pl.col('bolus').cast(pl.Float32),
    pl.col('age').cast(pl.Float32),
    pl.col('weight').cast(pl.Float32),
])

# --- Velocity: CGM - CGM_lag_1 ---
print("   📈 Velocity...")
q = q.with_columns(
    (pl.col('CGM') - pl.col('CGM_lag_1')).cast(pl.Float32).alias('velocity')
)

# --- Acceleration: Velocity - Velocity_lag_1 ---
print("   📈 Acceleration...")
q = q.with_columns(
    pl.col('velocity').shift(1).over('id').alias('velocity_lag_1')
)
q = q.with_columns(
    (pl.col('velocity') - pl.col('velocity_lag_1')).cast(pl.Float32).alias('acceleration')
)

# --- Rolling Mean (30 min = 6 periods) ---
print("   📊 Rolling Mean (30 min)...")
q = q.with_columns(
    pl.col('CGM').rolling_mean(window_size=6).over('id').cast(pl.Float32).alias('rolling_mean')
)

# --- Rolling Std (1 hr = 12 periods) ---
print("   📊 Rolling Std (1 hr)...")
q = q.with_columns(
    pl.col('CGM').rolling_std(window_size=12).over('id').cast(pl.Float32).alias('rolling_std')
)

# --- Active Insulin / IOB Proxy (2 hr = 24 periods) ---
print("   💉 Active Insulin (2 hr rolling sum)...")
q = q.with_columns(
    pl.col('bolus').rolling_sum(window_size=24).over('id').cast(pl.Float32).alias('active_insulin')
)

# --- Time Features ---
print("   ⏰ Time features...")
q = q.with_columns([
    pl.col('date').dt.hour().alias('hour'),
])
q = q.with_columns([
    (2 * np.pi * pl.col('hour') / 24).sin().cast(pl.Float32).alias('time_sin'),
    (2 * np.pi * pl.col('hour') / 24).cos().cast(pl.Float32).alias('time_cos'),
])

# --- Gender Encoding ---
print("   🏷️ Gender encoding...")
if 'gender' in keep:
    q = q.with_columns(
        pl.when(pl.col('gender').str.to_lowercase() == 'male').then(1.0)
        .when(pl.col('gender').str.to_lowercase() == 'female').then(0.0)
        .otherwise(-1.0).cast(pl.Float32).alias('gender_encoded')
    )
else:
    q = q.with_columns(pl.lit(-1.0).cast(pl.Float32).alias('gender_encoded'))

# --- TARGET: CGM_future (shift -6 = 30 min ahead) ---
print("   🎯 Target (CGM_future = shift -6)...")
q = q.with_columns(
    pl.col('CGM').shift(-6).over('id').cast(pl.Float32).alias('CGM_future')
)

# ============================================================
# STEP D: FINAL CLEANUP
# ============================================================
print("\n📋 Step D: Final cleanup...")

# Filter out null targets
q = q.filter(pl.col('CGM_future').is_not_null())

# Select final columns
q = q.select(FINAL_FEATURES)

# Drop any remaining nulls
q = q.drop_nulls()

# ============================================================
# STEP 2: EXECUTION (Stream to Disk)
# ============================================================
print(f"\n💾 Streaming to: {OUTPUT_TRAIN}")
print("   ⏳ This may take a few minutes...")

q.sink_parquet(OUTPUT_TRAIN)

print("\n" + "="*70)
print("✅ PROCESSING COMPLETE")
print("="*70)

In [ ]:
# ============================================================
# STATISTICS: Show cleaning impact
# ============================================================
print("\n📊 CLEANING STATISTICS")
print("-"*50)

if os.path.exists(OUTPUT_TRAIN):
    # Get final count
    final_count = pl.scan_parquet(OUTPUT_TRAIN).select(pl.len()).collect().item()
    file_size = os.path.getsize(OUTPUT_TRAIN) / (1024**3)
    
    dropped = original_count - final_count
    drop_pct = (dropped / original_count) * 100
    
    print(f"  Original rows:  {original_count:,}")
    print(f"  Final rows:     {final_count:,}")
    print(f"  Dropped rows:   {dropped:,} ({drop_pct:.2f}%)")
    print(f"  File size:      {file_size:.2f} GB")
    
    # Show sample
    print(f"\n📋 Sample data:")
    sample = pl.read_parquet(OUTPUT_TRAIN, n_rows=5)
    print(sample)
    
    # Show columns
    print(f"\n📋 Final columns: {sample.columns}")
else:
    print("❌ Output file not found!")

## Process TEST Data (Same Pipeline)

In [ ]:
# Process TEST data with same pipeline
RAW_TEST = f"{BASE_PATH}/raw/metabonet_public_test.parquet"
OUTPUT_TEST = f"{BASE_PATH}/processed/test_ready_for_glucose_prediction.parquet"

print("\n" + "="*70)
print("🔧 PROCESSING TEST DATA")
print("="*70)

# Same pipeline for test
q_test = pl.scan_parquet(RAW_TEST)
test_original = pl.scan_parquet(RAW_TEST).select(pl.len()).collect().item()
print(f"   Original rows: {test_original:,}")

available_test = q_test.collect_schema().names()
keep_test = [c for c in KEEP_COLS if c in available_test]
q_test = q_test.select(keep_test)

q_test = q_test.sort(['id', 'date'])

q_test = q_test.with_columns([
    pl.col('CGM').fill_null(strategy='forward').over('id'),
    pl.col('bolus').fill_null(0.0),
    pl.col('carbs').fill_null(0.0),
    pl.col('basal').fill_null(0.0),
    pl.col('age').fill_null(-1.0),
    pl.col('weight').fill_null(-1.0),
])

q_test = q_test.with_columns([pl.col('CGM').shift(1).over('id').alias('CGM_lag_1')])
q_test = q_test.with_columns([(pl.col('CGM') - pl.col('CGM_lag_1')).abs().alias('velocity_abs')])

q_test = q_test.filter(pl.col('CGM').is_not_null())
q_test = q_test.filter((pl.col('CGM') > CGM_MIN) & (pl.col('CGM') < CGM_MAX))
q_test = q_test.filter((pl.col('velocity_abs') < MAX_VELOCITY) | pl.col('velocity_abs').is_null())

q_test = q_test.with_columns([pl.col('CGM').cast(pl.Float32), pl.col('bolus').cast(pl.Float32), pl.col('age').cast(pl.Float32), pl.col('weight').cast(pl.Float32)])
q_test = q_test.with_columns((pl.col('CGM') - pl.col('CGM_lag_1')).cast(pl.Float32).alias('velocity'))
q_test = q_test.with_columns(pl.col('velocity').shift(1).over('id').alias('velocity_lag_1'))
q_test = q_test.with_columns((pl.col('velocity') - pl.col('velocity_lag_1')).cast(pl.Float32).alias('acceleration'))
q_test = q_test.with_columns(pl.col('CGM').rolling_mean(window_size=6).over('id').cast(pl.Float32).alias('rolling_mean'))
q_test = q_test.with_columns(pl.col('CGM').rolling_std(window_size=12).over('id').cast(pl.Float32).alias('rolling_std'))
q_test = q_test.with_columns(pl.col('bolus').rolling_sum(window_size=24).over('id').cast(pl.Float32).alias('active_insulin'))
q_test = q_test.with_columns([pl.col('date').dt.hour().alias('hour')])
q_test = q_test.with_columns([(2*np.pi*pl.col('hour')/24).sin().cast(pl.Float32).alias('time_sin'), (2*np.pi*pl.col('hour')/24).cos().cast(pl.Float32).alias('time_cos')])

if 'gender' in keep_test:
    q_test = q_test.with_columns(pl.when(pl.col('gender').str.to_lowercase()=='male').then(1.0).when(pl.col('gender').str.to_lowercase()=='female').then(0.0).otherwise(-1.0).cast(pl.Float32).alias('gender_encoded'))
else:
    q_test = q_test.with_columns(pl.lit(-1.0).cast(pl.Float32).alias('gender_encoded'))

q_test = q_test.with_columns(pl.col('CGM').shift(-6).over('id').cast(pl.Float32).alias('CGM_future'))
q_test = q_test.filter(pl.col('CGM_future').is_not_null())
q_test = q_test.select(FINAL_FEATURES)
q_test = q_test.drop_nulls()

print(f"💾 Streaming to: {OUTPUT_TEST}")
q_test.sink_parquet(OUTPUT_TEST)

if os.path.exists(OUTPUT_TEST):
    test_final = pl.scan_parquet(OUTPUT_TEST).select(pl.len()).collect().item()
    print(f"\n✅ Test: {test_original:,} → {test_final:,} rows")

print("\n🎉 ALL DATA PROCESSED!")